# Expensive process run with patient

In [0]:
from openai import OpenAI
import os

class Embeddings:
    def __init__(self):
        self.client = OpenAI(
            api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
             base_url="https://dbc-65fcd381-2e74.cloud.databricks.com/serving-endpoints"
             )

    def get_embeddings(self, text):
        return self.client.embeddings.create(
            input=text,
            model="databricks-bge-large-en"
        )
# embeddings = Embeddings().get_embeddings("This is a test")
# print(len(embeddings.data[0].embedding))

In [0]:
import glob
import json
import os
from datetime import datetime

def get_all_files():
    """
    Retrieves a list of all JSON files from a predefined directory.

    The directory path is hardcoded to '/Volumes/careconnect/default/input_data/'.
    
    Returns:
        list: A list of file paths (strings) for all .json files found.
              Returns an empty list if no files are found or if an error occurs.
    
    Raises:
        Prints an error message to the console if any unexpected error occurs
        during file globbing.
    """
    try:
        files = glob.glob('/Volumes/careconnect/default/input_data/*.json')
        return files
    except Exception as e:
        print(f"Error accessing directory or globbing files: {e}")
        return []

def generate_unique_chunk_id():
    """
    Generates a unique string ID based on the current datetime.

    The format is YYYYMMDDHHMMSSffffff (YearMonthDayHourMinuteSecondMicrosecond).

    Returns:
        str: A unique string identifier.
    
    Raises:
        Prints an error message to the console if datetime operations fail.
    """
    try:
        return datetime.now().strftime('%Y%m%d%H%M%S%f')
    except Exception as e:
        print(f"Error generating unique chunk ID: {e}")
        return "error_generating_id"

def read_files(data, file_name, embeddings):
    """
    Processes a list of data chunks, transforming them into a standardized format.

    Each chunk is augmented with a new unique chunk_id (original chunk_id + timestamp),
    page_number, and hospital_name (derived from the file_name).
    The remaining content of the chunk is converted to a JSON string.

    Args:
        data (list): A list of dictionaries, where each dictionary is an input data chunk.
                     Expected keys in each dictionary: 'chunk_id', 'page_number'.
        file_name (str): The path of the file from which the data was read.
                         Used to derive the hospital_name.

    Returns:
        list: A list of dictionaries, where each dictionary is a processed file_chunk.
              Returns an empty list if input data is invalid or an error occurs.
    
    Raises:
        Prints an error message to the console for various potential issues like
        invalid input data type, missing keys, or JSON serialization errors.
    """
    if not isinstance(data, list):
        print(f"Error: Input 'data' for file '{file_name}' is not a list.")
        return []

    file_data = []
    try:
        hospital_name_base = os.path.basename(file_name)
        hospital_name = os.path.splitext(hospital_name_base)[0].lower()
    except Exception as e:
        print(f"Error deriving hospital name from '{file_name}': {e}")
        hospital_name = "unknown_hospital"

    for i in data:
        if not isinstance(i, dict):
            print(f"Warning: Skipping non-dictionary item in data for file '{file_name}'. Item: {i}")
            continue
        
        file_chunk = {}
        try:
            original_chunk_id = i.get('chunk_id', 'unknown_id')
            file_chunk['chunk_id'] = f"{original_chunk_id}_{generate_unique_chunk_id()}"
            file_chunk['page_number'] = i.get('page_number', None)
            file_chunk['hospital_name'] = hospital_name
            
            content = i.copy()
            keys_to_remove = ['chunk_id', 'page_number', 'hospital_name', 'type', 
                              'hindi_message_title', 'hindi_message']
            for key in keys_to_remove:
                content.pop(key, None)
            
            file_chunk['text'] = json.dumps(content)
            file_chunk['embedding'] = embeddings.get_embeddings(file_chunk['text']).data[0].embedding
            file_data.append(file_chunk)
        except KeyError as ke:
            print(f"Warning: Missing key '{ke}' in chunk from file '{file_name}'. Chunk: {i}")
            continue
        except TypeError as te:
            print(f"Error serializing content to JSON for chunk '{original_chunk_id}' in file '{file_name}': {te}")
            continue
        except Exception as e:
            print(f"Unexpected error processing chunk from file '{file_name}': {e}. Chunk: {i}")
            continue 
            
    return file_data

def main():
    """
    Main function to orchestrate the processing of JSON files.

    It gets all JSON files from the specified directory, reads each file,
    processes its content, and prints the first two processed chunks for each file.
    """
    embeddings = Embeddings()

    files = get_all_files()
    if not files:
        print("No JSON files found to process.")
        return

    for file_path in files:
        
        print(f"Start processing file {file_path}")
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                file_content_str = f.read()
        except FileNotFoundError:
            print(f"Error: File not found at '{file_path}'. Skipping.")
            continue
        except IOError as e:
            print(f"Error reading file '{file_path}': {e}. Skipping.")
            continue
        except Exception as e:
            print(f"An unexpected error occurred while opening/reading '{file_path}': {e}. Skipping.")
            continue

        try:
            data_from_json = json.loads(file_content_str)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from file '{file_path}': {e}. Skipping.")
            continue
        except Exception as e:
            print(f"An unexpected error occurred while loading JSON from '{file_path}': {e}. Skipping.")
            continue
            
        processed_data = read_files(data_from_json, file_path, embeddings)
        if not processed_data:
            print(f"No valid data found in file '{file_path}'. Skipping.")

if __name__ == "__main__":
    main()